In [2]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [3]:
from langchain_groq import ChatGroq
groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant")


In [5]:
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [11]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate

from langchain_text_splitters import RecursiveCharacterTextSplitter




In [14]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

4️⃣ bs_kwargs (MOST IMPORTANT PART)
bs_kwargs=dict()


This tells WebBaseLoader:

“When parsing HTML, use BeautifulSoup with these rules.”

5️⃣ parse_only=bs4.SoupStrainer(...)
parse_only=bs4.SoupStrainer(


🔥 This is the key optimization

SoupStrainer tells BeautifulSoup:

“Only parse these parts of the HTML — ignore everything else.”

Benefits:

Faster parsing

Cleaner text

Better embeddings

Less noise in retrieval

6️⃣ class_=(...)
class_=("post-content","post-title","post-header")


This selects HTML elements by CSS class name.

It means:

“Only extract HTML elements whose class is one of these.”

So from the webpage, you keep:

🧠 post-content → main article text

🏷️ post-title → article title

🧾 post-header → metadata / headings

And you discard:

navigation bar

footer

related posts

comments

ads

In [15]:
import bs4
loader = WebBaseLoader(
    web_paths = ("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content","post-title","post-header")
        )
    ),
)

In [16]:
docs = loader.load()

it is better to break the entire document into chunks - cox all llm models have some kind of context size 

In [17]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splits=text_splitter.split_documents(docs)
vectorstore = Chroma.from_documents(documents=splits,embedding=embeddings) #storing in vectorstore db
retriever = vectorstore.as_retriever()


In [21]:
#prompt template 
system_prompt = """you are an assistant for question-answering tasks.
    Use the following pieces of retrieved context to answer the question
    If you don't know the answer, say that you don't know. Use three sentences maximum and keep the answer concise
    \n\n
    {context}"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}")
    ]
)

In [22]:
from langchain_core.runnables import RunnablePassthrough

LangChain does:

retriever.invoke("What is an agent?")

Result → list of Document

Put into {context}

Pass question unchanged to {input}

Format prompt

Call LLM

👉 Retriever is clearly visible and explicit

RunnablePassthrough() means: “don’t change the input — just forward it as-is.”

In [23]:
rag_chain = {"context":retriever,"input":RunnablePassthrough()}|prompt|llm

In [24]:
response = rag_chain.invoke("what is self reflection?")

### Adding Chat History 

In [40]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
contextualize_q_system_prompt = (
    """
    Given a chat history and the latest user question which might reference context in the chat history, formulate a standalone question which can be understood without the chat history. Do NOT answer the question,just reformulate it if needed and otherwise return it as is.
    """
)
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
    ("system",contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}"),
    ]


)


✅ Correct naming (this matters in LangChain)
contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)


💡 Rule

If a prompt contains MessagesPlaceholder("chat_history"),
it is NOT a QA prompt.

🧠 What this prompt ACTUALLY does

Given:

Chat history:
User: What is self reflection?
AI: explanation...

New question:
"tell me more about it"


This prompt turns:

"tell me more about it"


into something like:

"Tell me more about self reflection"


✔ No retrieval
✔ No answering
✔ Only rewriting

✅ Where this prompt should be used

ONLY HERE 👇

history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    contextualize_q_prompt
)


That’s it.
🚫 Do NOT pipe this prompt to the LLM directly later.

❌ Where it should NOT be used
| qa_prompt | llm   ❌ WRONG


Because:

This prompt doesn’t contain {context}

It explicitly says “Do NOT answer the question”

✅ You STILL need a real QA prompt

Example:

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer the question using ONLY the provided context."),
        ("human", "{input}\n\nContext:\n{context}")
    ]
)


🚫 No chat_history here
🚫 No rewriting instructions

🧩 Final correct architecture (visual)
(chat_history, input)
        ↓
contextualize_q_prompt
        ↓
standalone question
        ↓
retriever
        ↓
context
        ↓
qa_prompt
        ↓
LLM answer

In [41]:
history_aware_retriever = create_history_aware_retriever(llm,retriever,contextualize_q_prompt)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7fc09c4b9a20>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChu

In [42]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer the question using ONLY the provided context."),
        ("human", "{input}\n\nContext:\n{context}")
    ]
)


In [43]:
rag_chain = {"context":history_aware_retriever,"input":RunnablePassthrough()}|qa_prompt|llm

In [46]:

from langchain_core.messages import AIMessage,HumanMessage
chat_history = []
question="what is self reflection?"
response = rag_chain.invoke(
    {
        "input": question,
        "chat_history": chat_history
    }
)
chat_history.extend(
    [
        HumanMessage(content=question),
        response
    ]
)
question2 = "tell me more about it"
response2 = response = rag_chain.invoke(
    {
        "input": question2,
        "chat_history": chat_history
    }
)
chat_history.extend(
    [
        HumanMessage(content=question2),
      response2
    ]
)

1️⃣ Your chain (annotated)
rag_chain = {
    "context": history_aware_retriever,
    "input": RunnablePassthrough()
} | qa_prompt | llm

What flows through the chain?

Input you pass to .invoke():

{
    "input": question,
    "chat_history": chat_history
}

2️⃣ What history_aware_retriever actually does
"context": history_aware_retriever

Step-by-step:

Takes:

input

chat_history

Uses LLM + contextualize prompt to:

Rewrite follow-ups like:

“tell me more about it”

Into:

“Tell me more about self reflection”

Sends that rewritten question to:

retriever

Outputs:

Documents (list of Document objects)

👉 Important
history_aware_retriever returns documents, NOT text.

So at this point:

context = [Document(...), Document(...)]

3️⃣ Why qa_prompt ALSO needs {context}

Your QA prompt:

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer the question using ONLY the provided context."),
        ("human", "{input}\n\nContext:\n{context}")
    ]
)

This does NOT retrieve anything ❌

It only formats the prompt.

Think of it like this:

Component	Responsibility
Retriever	Find relevant info
QA Prompt	Inject that info into the LLM
LLM	Generate answer

Without {context} in the prompt:

The LLM would never see the retrieved documents

Retrieval would be wasted

4️⃣ So is context “used twice”?

❌ No — it is passed, then consumed

Timeline:

1️⃣ Retriever produces context
2️⃣ Prompt consumes context
3️⃣ LLM reasons over context

In [47]:
chat_history 

[HumanMessage(content='what is self reflection?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Self-reflection is a vital aspect that allows autonomous agents to improve iteratively by refining past action decisions and correcting previous mistakes. It plays a crucial role in real-world tasks where trial and error are inevitable. Self-reflection is created by showing two-shot examples to a Large Language Model (LLM) and each example is a pair of (failed trajectory, ideal reflection for guiding future changes in the plan).', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 1006, 'total_tokens': 1088, 'completion_time': 0.091932065, 'completion_tokens_details': None, 'prompt_time': 0.072406966, 'prompt_tokens_details': None, 'queue_time': 0.012725243, 'total_time': 0.164339031}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs':

In [48]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
store = {}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer"
)

In [57]:
reponse_dict=conversational_rag_chain.invoke(
    {"input":"what is task decomposition?"},
        config={
            "configurable":{"session_id":"user123"},
        
    },
)

Error in RootListenersTracer.on_chain_end callback: KeyError('answer')


In [58]:
reponse_dict=conversational_rag_chain.invoke(
    {"input":"what are common ways of doing it "}, #doing task decomposion, it will get context acc to prev chat history 
        config={
            "configurable":{"session_id":"user123"},
            "return_as_dict":True
        ,
    },
)

Error in RootListenersTracer.on_chain_end callback: KeyError('answer')


In [59]:
print(reponse_dict.content)

The common ways of doing it, as mentioned in the provided context, include:

1. Task planning using Large Language Models (LLM) to parse user requests into multiple tasks.
2. Task decomposition using techniques like Chain of Thought (CoT) and Tree of Thoughts (Yao et al. 2023) to break down complex tasks into smaller and simpler steps.
3. Fine-tuning with techniques like Consecutive Hint (CoH) to present a history of sequentially improved outputs and train the model to take on the trend to produce better outputs.
